# Spike Density Function — All Neurons with Classification

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import scipy.io
import warnings
from notebooks.imports import *
from scipy.ndimage import gaussian_filter1d
from config import dir_config

compiled_dir = Path(dir_config.data.compiled)
sorting_dir = Path(dir_config.data.sorting)
processed_dir = Path(dir_config.data.processed)

### Load neuron metadata

In [ ]:
classification_df = pd.read_csv(Path(processed_dir, "neuron_classification.csv"))
neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))

In [ ]:
classification_df

### Alignment settings

In [ ]:
alignment_settings_VGS = [
    {"alignment_event": "target_onset",   "start_time_ms": -100, "end_time_ms": 300},
    {"alignment_event": "response_onset", "start_time_ms": -300, "end_time_ms": 100},
]

alignment_settings_GP = [
    {"alignment_event": "target_onset",   "start_time_ms": -200, "end_time_ms": 300},
    {"alignment_event": "stimulus_onset", "start_time_ms": -100, "end_time_ms": 800},
    {"alignment_event": "response_onset", "start_time_ms": -300, "end_time_ms": 50},
]

alignment_buffer = 50

# which trial comparisons to plot
trial_types_to_plot = ["VGS_to_away", "GP_to_away"]

---
### Helper functions (SDF computation, plotting, trial extraction)

In [ ]:
def _load_array(path_stem):
    """Load a numpy array from .npy or .mat file, whichever exists."""
    npy_path = Path(str(path_stem) + ".npy")
    mat_path = Path(str(path_stem) + ".mat")
    if npy_path.exists():
        return np.load(npy_path)
    mat = scipy.io.loadmat(mat_path)
    key = next(k for k in mat if not k.startswith("_"))
    return mat[key].squeeze()


def _spike_train_convolved(cluster_spike_time, timestamps, valid_trial, alignment_settings, alignment_buffer):
    valid_trial_timestamps = timestamps.iloc[valid_trial]
    spike_train = []
    convolved   = []
    sigma = 10

    for s in alignment_settings:
        n_bins = 1 + s["end_time_ms"] - s["start_time_ms"] + 2 * alignment_buffer
        st = np.zeros([valid_trial.shape[0], n_bins])
        cv = np.zeros([valid_trial.shape[0], n_bins])

        for idx, trial_id in enumerate(valid_trial):
            t0 = valid_trial_timestamps.loc[trial_id, s["alignment_event"]]
            start_ts = t0 + (s["start_time_ms"] - alignment_buffer) * 30
            end_ts   = t0 + (s["end_time_ms"]   + alignment_buffer) * 30
            spks = cluster_spike_time[(cluster_spike_time >= start_ts) & (cluster_spike_time <= end_ts)] - start_ts
            st[idx, np.ceil(spks / 30).astype(int)] = 1
            cv[idx] = gaussian_filter1d(st[idx], sigma=sigma, truncate=3)

            if s["alignment_event"] == "stimulus_onset":
                resp_ts = valid_trial_timestamps.loc[trial_id, "response_onset"]
                if (end_ts - alignment_buffer * 30) > resp_ts - 50 * 30:
                    pre_sac = np.ceil((end_ts - resp_ts) / 30 + 50 - alignment_buffer).astype(int)
                    st[idx, -pre_sac:] = np.nan
                    cv[idx, -pre_sac:] = np.nan

        spike_train.append(st[:, alignment_buffer:-alignment_buffer])
        convolved.append(cv[:, alignment_buffer:-alignment_buffer] * 1000)

    return spike_train, convolved


def _plot_VGS(axs, fig_idx, title, alignment_settings, convolved, trial_types):
    for ai, s in enumerate(alignment_settings):
        ax = axs[ai, fig_idx]
        for tt in trial_types:
            x = range(s["start_time_ms"], s["end_time_ms"] + 1)
            y = convolved[ai][tt["trial_number"], :].mean(axis=0)
            ax.plot(x, y, color=tt["color"], linewidth=tt["linewidth"], linestyle=tt["linestyle"], label=tt["label"])
            if tt["shade_alpha"] > 0:
                ye = convolved[ai][tt["trial_number"], :].std(axis=0) / np.sqrt(len(tt["trial_number"]))
                ax.fill_between(x, y + ye, y - ye, alpha=tt["shade_alpha"], edgecolor=tt["color"], facecolor=tt["color"])
        if fig_idx == 0:
            ax.set_ylabel(s["alignment_event"])
        if ai == 0:
            ax.set_title(title)
        ax.axvline(x=0, linestyle="--", color="k")
        ax.legend()


def _plot_GP(axs, fig_idx, title, alignment_settings, convolved, coherence_levels, GP_valid_trial_idx, trial_info, trial_types, plot_std):
    colors = ["#e31a1c", "#ff7f00", "#33a02c", "#1f78b4"]
    for ai, s in enumerate(alignment_settings):
        ax = axs[ai, fig_idx]
        for tt in trial_types:
            for ci, coh in enumerate(coherence_levels):
                tn = tt["trial_number"][ci]
                if len(tn) == 0:
                    continue
                if s["alignment_event"] == "stimulus_onset":
                    rt  = np.nanmedian(trial_info.loc[GP_valid_trial_idx[tn], "reaction_time"])
                    end = int(np.min([rt - 50, s["end_time_ms"]])) - s["start_time_ms"] + 2
                else:
                    end = s["end_time_ms"] - s["start_time_ms"] + 2
                x = range(s["start_time_ms"], s["end_time_ms"] + 1)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    y = np.nanmean(convolved[ai][tn, :], axis=0)
                y[end:] = np.nan
                lbl = str(coh) if tt["label"] else None
                ax.plot(x, y, color=colors[ci], linewidth=tt["linewidth"], linestyle=tt["linestyle"], label=lbl)
                if plot_std:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", RuntimeWarning)
                        ye = np.nanstd(convolved[ai][tn, :], axis=0) / np.sqrt(len(tn))
                    ye[end:] = np.nan
                    ax.fill_between(x, y + ye, y - ye, alpha=tt["shade_alpha"], edgecolor=colors[ci], facecolor=colors[ci])
        if fig_idx == 0:
            ax.set_ylabel(s["alignment_event"])
        if ai == 0:
            ax.set_title(title)
        ax.axvline(x=0, linestyle="--", color="k")
        ax.legend()


# --- trial extraction helpers ---

def _VGS_to_away(trial_info, idx):
    to  = np.where(trial_info.loc[idx, "choice"] == 1)[0]
    aw  = np.where(trial_info.loc[idx, "choice"] == 0)[0]
    return [{"trial_number": to, "linewidth": 2, "linestyle": "-",  "color": "k", "label": "toRF",  "shade_alpha": 0.5},
            {"trial_number": aw, "linewidth": 1, "linestyle": "--", "color": "k", "label": "awayRF", "shade_alpha": 0}]


def _VGS_pre_post_to_away(trial_info, idx):
    gp_start = np.where(trial_info["task_type"] == 1)[0][0]
    gp_end   = np.where(trial_info["task_type"] == 1)[0][-1]
    pre_to  = np.where((trial_info.loc[idx, "choice"] == 1) & (idx < gp_start))[0]
    pre_aw  = np.where((trial_info.loc[idx, "choice"] == 0) & (idx < gp_start))[0]
    post_to = np.where((trial_info.loc[idx, "choice"] == 1) & (idx > gp_end))[0]
    post_aw = np.where((trial_info.loc[idx, "choice"] == 0) & (idx > gp_end))[0]
    return [{"trial_number": pre_to,  "linewidth": 2, "linestyle": "-",  "color": [.5,.5,.5], "label": "pre_GP",  "shade_alpha": 0.5},
            {"trial_number": pre_aw,  "linewidth": 1, "linestyle": "--", "color": [.5,.5,.5], "label": None,      "shade_alpha": 0},
            {"trial_number": post_to, "linewidth": 2, "linestyle": "-",  "color": "k",        "label": "post_GP", "shade_alpha": 0.5},
            {"trial_number": post_aw, "linewidth": 1, "linestyle": "--", "color": "k",        "label": None,      "shade_alpha": 0}]


def _coh_trials(coherence_levels, trial_info, idx, choice, prob=None, outcome_req=True):
    """Return per-coherence trial index list for given choice (1=toRF, 0=awayRF) and optional prob_toRF filter."""
    result = []
    for coh in coherence_levels:
        mask = (trial_info.loc[idx, "choice"] == choice) & (trial_info.loc[idx, "coherence"] == coh)
        if coh != 0 and outcome_req:
            mask &= trial_info.loc[idx, "outcome"] == 1
        if prob == "equal":
            mask &= trial_info.loc[idx, "prob_toRF"] == 50
        elif prob == "unequal":
            mask &= trial_info.loc[idx, "prob_toRF"] != 50
        result.append(np.where(mask)[0])
    return result


def _GP_to_away(cohs, trial_info, idx):
    return [{"trial_number": _coh_trials(cohs, trial_info, idx, 1), "linewidth": 2, "linestyle": "-",  "label": True,  "shade_alpha": 0.6},
            {"trial_number": _coh_trials(cohs, trial_info, idx, 0), "linewidth": 1, "linestyle": "--", "label": False, "shade_alpha": 0.2}]


def _GP_to_equal_unequal(cohs, trial_info, idx):
    return [{"trial_number": _coh_trials(cohs, trial_info, idx, 1, prob="equal"),   "linewidth": 1,   "linestyle": "-",  "label": True,  "shade_alpha": 0},
            {"trial_number": _coh_trials(cohs, trial_info, idx, 1, prob="unequal"), "linewidth": 2.5, "linestyle": "-",  "label": False, "shade_alpha": 0}]


def _GP_away_equal_unequal(cohs, trial_info, idx):
    return [{"trial_number": _coh_trials(cohs, trial_info, idx, 0, prob="equal"),   "linewidth": 0.5, "linestyle": "--", "label": True,  "shade_alpha": 0},
            {"trial_number": _coh_trials(cohs, trial_info, idx, 0, prob="unequal"), "linewidth": 1.5, "linestyle": "--", "label": False, "shade_alpha": 0}]


def _GP_to_away_equal(cohs, trial_info, idx):
    return [{"trial_number": _coh_trials(cohs, trial_info, idx, 1, prob="equal"), "linewidth": 2, "linestyle": "-",  "label": True,  "shade_alpha": 0.6},
            {"trial_number": _coh_trials(cohs, trial_info, idx, 0, prob="equal"), "linewidth": 1, "linestyle": "--", "label": False, "shade_alpha": 0.2}]


def _GP_to_away_unequal(cohs, trial_info, idx):
    return [{"trial_number": _coh_trials(cohs, trial_info, idx, 1, prob="unequal"), "linewidth": 2, "linestyle": "-",  "label": True,  "shade_alpha": 0.6},
            {"trial_number": _coh_trials(cohs, trial_info, idx, 0, prob="unequal"), "linewidth": 1, "linestyle": "--", "label": False, "shade_alpha": 0.2}]


### Plot — all neurons

### Export to PowerPoint — one slide per neuron

In [ ]:
import tempfile
from PIL import Image as PILImage
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN

qc_pptx_path  = Path(processed_dir, "qc_all_neurons.pptx")
out_pptx_path = Path(processed_dir, "qc_all_neurons.pptx")

SLIDE_W   = 13.33   # inches
SLIDE_H   = 7.5
TITLE_H   = 0.55
IMG_TOP   = TITLE_H + 0.1
AVAIL_H   = SLIDE_H - IMG_TOP
MIN_YLIM  = 30

def _apply_ylim_floor(fig, min_upper=MIN_YLIM):
    for ax in fig.axes:
        lo, hi = ax.get_ylim()
        if hi < min_upper:
            ax.set_ylim(lo, min_upper)

def _place_image(slide, img_path, left_in, col_w_in, avail_h_in, top_in):
    with PILImage.open(img_path) as im:
        w_px, h_px = im.size
    aspect = w_px / h_px
    img_w = col_w_in
    img_h = col_w_in / aspect
    if img_h > avail_h_in:
        img_h = avail_h_in
        img_w = avail_h_in * aspect
    left_centered = left_in + (col_w_in - img_w) / 2
    slide.shapes.add_picture(
        str(img_path),
        Inches(left_centered), Inches(top_in),
        width=Inches(img_w), height=Inches(img_h),
    )

def _move_slide(prs, from_idx, to_idx):
    """Move slide at from_idx to to_idx by manipulating the slide ID list."""
    sld_id_lst = prs.slides._sldIdLst
    items = list(sld_id_lst)
    item = items[from_idx]
    sld_id_lst.remove(item)
    sld_id_lst.insert(to_idx, item)

# Open existing QC pptx — slides 0..n-1 are the original QC slides
prs = Presentation(qc_pptx_path)
n_qc = len(list(prs.slides))
assert n_qc == len(neuron_metadata), (
    f"QC pptx has {n_qc} slides but neuron_metadata has {len(neuron_metadata)} rows"
)
blank_layout = prs.slide_layouts[6]

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    for session_name, session_neurons in neuron_metadata.groupby("session_id", sort=False):
        spike_times    = _load_array(Path(compiled_dir, session_name, "spike_times"))
        spike_clusters = _load_array(Path(compiled_dir, session_name, "spike_clusters"))
        timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps_cleaned.csv"), index_col=None)
        trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial_cleaned.csv"),      index_col=None)
        coherence_levels = np.unique(trial_info.loc[~np.isnan(trial_info["coherence"]), "coherence"])
        VGS_valid_trial_idx = np.where((trial_info["task_type"] == 2) & (trial_info["outcome"] == 1) & (~np.isnan(trial_info["reaction_time"])))[0]
        GP_valid_trial_idx  = np.where((trial_info["task_type"] == 1) & (trial_info["outcome"] >= 0) & (~np.isnan(trial_info["reaction_time"])))[0]

        for _, row in session_neurons.iterrows():
            cluster_id     = int(row["cluster"])
            classification = classification_df.loc[classification_df.neuron_id == row["neuron_id"], "classification"].values[0]
            neuron_id      = row["neuron_id"]
            slide_idx      = neuron_metadata.index.get_loc(row.name)

            cluster_spike_time = spike_times[spike_clusters == cluster_id]
            VGS_spike_train, VGS_convolved = _spike_train_convolved(
                cluster_spike_time, timestamps, VGS_valid_trial_idx, alignment_settings_VGS, alignment_buffer)
            GP_spike_train, GP_convolved = _spike_train_convolved(
                cluster_spike_time, timestamps, GP_valid_trial_idx, alignment_settings_GP, alignment_buffer)

            title_base = f"{session_name} | neuron {neuron_id} | {classification}"

            # --- VGS figure ---
            vgs_items = [t for t in trial_types_to_plot if t.startswith("VGS")]
            vgs_path = None
            if vgs_items:
                fig, axs = plt.subplots(len(alignment_settings_VGS), len(vgs_items), sharey="row", layout="constrained")
                axs = np.array(axs).reshape(len(alignment_settings_VGS), len(vgs_items))
                for fig_idx, ttype in enumerate(vgs_items):
                    if ttype == "VGS_to_away":
                        _plot_VGS(axs, fig_idx, " VGS toRF / awayRF", alignment_settings_VGS, VGS_convolved,
                                  _VGS_to_away(trial_info, VGS_valid_trial_idx))
                    elif ttype == "VGS_pre_post_to_away":
                        _plot_VGS(axs, fig_idx, " VGS toRF/awayRF pre & post GP", alignment_settings_VGS, VGS_convolved,
                                  _VGS_pre_post_to_away(trial_info, VGS_valid_trial_idx))
                _apply_ylim_floor(fig)
                vgs_path = tmpdir / f"{neuron_id}_vgs.png"
                fig.savefig(vgs_path, dpi=150, bbox_inches="tight")
                plt.close(fig)

            # --- GP figure ---
            gp_items = [t for t in trial_types_to_plot if t.startswith("GP")]
            gp_path = None
            if gp_items:
                fig, axs = plt.subplots(len(alignment_settings_GP), len(gp_items), sharey="row", layout="constrained")
                axs = np.array(axs).reshape(len(alignment_settings_GP), len(gp_items))
                fig.set_size_inches([10, 10])
                for fig_idx, ttype in enumerate(gp_items):
                    if ttype == "GP_to_away":
                        _plot_GP(axs, fig_idx, " GP toRF / awayRF", alignment_settings_GP, GP_convolved,
                                 coherence_levels, GP_valid_trial_idx, trial_info,
                                 _GP_to_away(coherence_levels, trial_info, GP_valid_trial_idx), plot_std=True)
                    elif ttype == "GP_to_equal_unequal":
                        _plot_GP(axs, fig_idx, " GP toRF equal/unequal prior", alignment_settings_GP, GP_convolved,
                                 coherence_levels, GP_valid_trial_idx, trial_info,
                                 _GP_to_equal_unequal(coherence_levels, trial_info, GP_valid_trial_idx), plot_std=True)
                    elif ttype == "GP_away_equal_unequal":
                        _plot_GP(axs, fig_idx, " GP awayRF equal/unequal prior", alignment_settings_GP, GP_convolved,
                                 coherence_levels, GP_valid_trial_idx, trial_info,
                                 _GP_away_equal_unequal(coherence_levels, trial_info, GP_valid_trial_idx), plot_std=True)
                    elif ttype == "GP_to_away_equal":
                        _plot_GP(axs, fig_idx, " GP toRF/awayRF equal prior", alignment_settings_GP, GP_convolved,
                                 coherence_levels, GP_valid_trial_idx, trial_info,
                                 _GP_to_away_equal(coherence_levels, trial_info, GP_valid_trial_idx), plot_std=False)
                    elif ttype == "GP_to_away_unequal":
                        _plot_GP(axs, fig_idx, " GP toRF/awayRF unequal prior", alignment_settings_GP, GP_convolved,
                                 coherence_levels, GP_valid_trial_idx, trial_info,
                                 _GP_to_away_unequal(coherence_levels, trial_info, GP_valid_trial_idx), plot_std=False)
                _apply_ylim_floor(fig)
                gp_path = tmpdir / f"{neuron_id}_gp.png"
                fig.savefig(gp_path, dpi=150, bbox_inches="tight")
                plt.close(fig)

            # Append SDF slide at the end (slides n..2n-1 will be SDF slides)
            sdf_slide = prs.slides.add_slide(blank_layout)
            title_box = sdf_slide.shapes.add_textbox(
                Inches(0.2), Inches(0.05), Inches(SLIDE_W - 0.4), Inches(TITLE_H))
            tf = title_box.text_frame
            tf.text = title_base
            tf.paragraphs[0].alignment = PP_ALIGN.CENTER
            tf.paragraphs[0].runs[0].font.size = Pt(16)
            tf.paragraphs[0].runs[0].font.bold = True
            both = [p for p in [vgs_path, gp_path] if p is not None]
            col_w = SLIDE_W / len(both)
            for k, img_path in enumerate(both):
                _place_image(sdf_slide, img_path, k * col_w, col_w, AVAIL_H, IMG_TOP)

            print(f"[{slide_idx+1}/{len(neuron_metadata)}] neuron {neuron_id} ({session_name}) done")

# Rearrange: slides 0..n-1 are QC, slides n..2n-1 are SDF.
# Move SDF slide i (at index n+i) to position 2*i, just before its QC slide.
# Each move keeps remaining SDF slides at n+j, so the formula is stable across iterations.
n = len(neuron_metadata)
for i in range(n):
    _move_slide(prs, n + i, 2 * i)

prs.save(out_pptx_path)
print(f"Saved → {out_pptx_path}  ({len(list(prs.slides))} slides total)")